# Tarea para el Hogar 04

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion sw Hiperparámetros lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.


Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [7]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Oct 23 16:04:23 2026"

In [8]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,672830,36.0,1479647,79.1,1479647,79.1
Vcells,1245862,9.6,8388608,64.0,1991727,15.2


### 2.2.2 Carga de Librerias

In [9]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

# Optuna se usa desde R: LightGBM y los datos permanecen en R.
# Ejecutar al iniciar la sesión R, antes de importar módulos de Python.
if (!requireNamespace("reticulate", quietly = TRUE) ||
    utils::packageVersion("reticulate") < "1.41.0") {
  install.packages("reticulate")
}
Sys.setenv(RETICULATE_PYTHON = "managed")
reticulate::py_require("optuna>=4.0,<6.0")
optuna <- reticulate::import("optuna", convert = TRUE)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm



### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [10]:
PARAM <- list()
# Resultados de esta prueba en carpetas distintas a las corridas anteriores.
PARAM$experimento <- "5949"
# Semilla utilizada en los experimentos que veníamos comparando.
PARAM$semilla_primigenia <- 149911

In [11]:
PARAM$kaggle$competencia <- "labo-1-virtual-inicial"
PARAM$kaggle$cortes <- seq(9000L, 14000L, by = 500L)

In [12]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5

In [13]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# Presupuesto para cinco dimensiones: 15 ensayos iniciales y luego TPE.
PARAM$hyperparametertuning$n_trials <- 80L
PARAM$hyperparametertuning$n_startup_trials <- 15L

# Cantidad fija en cada ensayo y en el entrenamiento final.
PARAM$hyperparametertuning$num_iterations <- 1000L

# Espacio inicial para aprendizaje, complejidad y regularización.
# Los rangos son un punto de partida, no una importancia medida en estos datos.
PARAM$hyperparametertuning$espacio <- list(
  learning_rate = c(0.01, 0.15),
  num_leaves = c(16L, 256L),
  min_data_in_leaf = c(20L, 2000L),
  feature_fraction = c(0.5, 1.0),
  bagging_fraction = c(0.6, 1.0)
)

# Parametros fijos que se sobrescriben con la grilla
PARAM$lgbm$param_fijos <- list(
  boosting = "gbdt",
  objective = "binary",
  metric = "auc",
  first_metric_only = FALSE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  force_row_wise = TRUE,
  verbosity = -100,

  seed = PARAM$semilla_primigenia,

  max_depth = -1L,
  min_gain_to_split = 0,
  min_sum_hessian_in_leaf = 0.001,
  lambda_l1 = 0.0,
  lambda_l2 = 0.0,
  max_bin = 31L,

  bagging_fraction = 1.0,
  pos_bagging_fraction = 1.0,
  neg_bagging_fraction = 1.0,
  is_unbalance = FALSE,
  scale_pos_weight = 1.0,

  drop_rate = 0.1,
  max_drop = 50,
  skip_drop = 0.5,

  extra_trees = FALSE,

  # Valores de referencia del experimento 5940
  num_iterations = 300L,
  learning_rate = 0.03,
  feature_fraction = 0.75,
  num_leaves = 64L,
  min_data_in_leaf = 500L
)

### 2.2.4  Preprocesamiento

In [14]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [15]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [16]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [17]:
# Etiqueta binaria de esta prueba: BAJA+2 = 1; BAJA+1 y CONTINUA = 0.
# El entrenamiento y el AUC de validación usan esta misma definición.
dataset_train[, clase01 := as.integer(clase_ternaria == "BAJA+2")]

In [18]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [19]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [20]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83665

[1] 154

### 2.2.5 Optimización bayesiana con Optuna: AUC de BAJA+2

Se reemplaza la grilla por **Optuna con TPESampler**. Cada ensayo propone
hiperparámetros, entrena LightGBM en R y devuelve a Optuna el AUC medio de
5 folds en la ronda 1000. Después de 15 ensayos iniciales aleatorios, TPE
utiliza los resultados anteriores para orientar las propuestas restantes.
El presupuesto total es de **80 ensayos**; no se recorre un producto cartesiano.

**Etiqueta:** `BAJA+2 = 1`; `BAJA+1` y `CONTINUA = 0`. Esa misma etiqueta
se utiliza en la validación y en el entrenamiento final. Se conservan
`objective = "binary"` y `metric = "auc"`: Optuna maximiza el AUC de
`BAJA+2` frente al resto, no la ganancia de la competencia.

Parámetros optimizados, configurables en `PARAM$hyperparametertuning$espacio`:

| Hiperparámetro | Rango | Escala |
| --- | --- | --- |
| `learning_rate` | 0.01 a 0.15 | logarítmica |
| `num_leaves` | 16 a 256 | entera logarítmica |
| `min_data_in_leaf` | 20 a 2000 | entera logarítmica |
| `feature_fraction` | 0.5 a 1.0 | lineal |
| `bagging_fraction` | 0.6 a 1.0 | lineal |

* **1000 iteraciones fijas** en CV y producción, sin early stopping ni pruning.
  No se usa el pico de AUC de una ronda anterior para seleccionar el ganador.
* `lambda_l1 = lambda_l2 = 0` quedan fuera de la búsqueda. `bagging_freq = 1`
  activa la fracción de bagging propuesta. `max_bin = 31` permanece fijo.
* Se reutilizan los mismos folds, reiniciando la semilla en cada evaluación.
  La semilla es **149911** y los cortes de envío son **9000 a 14000**, como
  en las corridas que veníamos comparando.
* Se conserva el muestreo original: todos los `BAJA+1` y `BAJA+2`, y el 50%
  de los `CONTINUA`. `BAJA+1` es ahora un negativo retenido al 100%.
  La CV sigue operando sobre esa muestra; esta variante no incorpora la
  validación completa de la versión que optimiza ganancia.
* Si la muestra es pequeña, se limita el máximo de `min_data_in_leaf` a
  un cuarto del tamaño aproximado del entrenamiento de cada fold. El rango
  efectivo queda registrado en `PARAM.yml`.

Se registra cada ensayo en `tb_optuna_AUC_BAJA2_1000.txt`; el ganador por
AUC pasa al entrenamiento completo y al flujo original de producción.
El ajuste original de `min_data_in_leaf` al entrenar sin undersampling se
conserva. Un AUC mayor de BAJA+2 no garantiza más ganancia entre los clientes
contactados: esa comparación se debe revisar por separado en validación.

In [21]:
# Evalúo la AUC de la ronda 1000, que es la que usará producción.
Estimar_AUC_lightgbm <- function(x) {
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)
  rondas <- as.integer(PARAM$hyperparametertuning$num_iterations)
  param_completo$num_iterations <- rondas

  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
  modelocv <- lgb.cv(
    data = dtrain,
    nrounds = rondas,
    nfold = PARAM$hyperparametertuning$xval_folds,
    stratified = TRUE,
    params = param_completo,
    early_stopping_rounds = NULL,
    record = TRUE
  )

  historia_auc <- unlist(modelocv$record_evals$valid$auc$eval, use.names = FALSE)
  rm(modelocv)
  gc(full = TRUE, verbose = FALSE)

  if (length(historia_auc) != rondas || any(!is.finite(historia_auc)) ||
      any(historia_auc < 0 | historia_auc > 1)) {
    stop("No se obtuvo una historia de AUC válida para todas las rondas.")
  }

  # best_score puede ser el pico de una ronda anterior. Para esta prueba
  # fijo la selección en la misma ronda que el entrenamiento final.
  AUC <- as.numeric(historia_auc[rondas])
  message(format(Sys.time(), "%a %b %d %X %Y  "), toString(x),
          " AUC en ronda ", rondas, ": ", AUC)

  return(list(AUC = AUC, num_iterations = rondas))
}

In [22]:
# Creo un estudio nuevo, secuencial y con semilla reproducible.
# TPE adapta las propuestas a los resultados de los ensayos anteriores.
stopifnot(
  PARAM$hyperparametertuning$n_trials > 0L,
  PARAM$hyperparametertuning$n_startup_trials > 0L,
  PARAM$hyperparametertuning$n_startup_trials <
    PARAM$hyperparametertuning$n_trials,
  PARAM$hyperparametertuning$num_iterations == 1000L
)

# Ajusto min_data_in_leaf al tamaño disponible en cada entrenamiento de CV.
# Con pocos datos evito que todo el rango produzca árboles de una sola hoja.
espacio <- copy(PARAM$hyperparametertuning$espacio)
n_train_cv <- floor(nrow(dtrain) *
  (PARAM$hyperparametertuning$xval_folds - 1) /
  PARAM$hyperparametertuning$xval_folds)
limite_hoja <- max(2L, as.integer(floor(n_train_cv / 4)))
espacio$min_data_in_leaf[2] <- min(espacio$min_data_in_leaf[2], limite_hoja)
espacio$min_data_in_leaf[1] <- min(espacio$min_data_in_leaf)
PARAM$hyperparametertuning$espacio_efectivo <- copy(espacio)

sampler <- optuna$samplers$TPESampler(
  seed = as.integer(PARAM$semilla_primigenia),
  n_startup_trials = as.integer(PARAM$hyperparametertuning$n_startup_trials),
  multivariate = FALSE
)

# Conservo el enum de Python: convert = TRUE lo transformaría en un entero
# y tell() necesita el objeto TrialState para registrar un error.
estado_fallido <- reticulate::import("optuna.trial", convert = FALSE)$TrialState$FAIL

estudio <- optuna$create_study(
  direction = "maximize",
  sampler = sampler,
  study_name = paste0("HT", PARAM$experimento)
)

# Conservo la tabla y agrego los parámetros seleccionados y las rondas fijas.
tb_nueva <- data.table(
  learning_rate = numeric(),
  num_leaves = integer(),
  min_data_in_leaf = integer(),
  feature_fraction = numeric(),
  bagging_fraction = numeric(),
  num_iterations = integer(),
  AUC = numeric()
)

In [23]:
# veo que tiene la tabla antes de procesar
tb_nueva

learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,num_iterations,AUC
<dbl>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>


In [24]:
# Optuna propone (ask), LightGBM evalúa en R y recibe la AUC (tell).
# No se ejecuta el producto cartesiano de una grilla.
espacio <- PARAM$hyperparametertuning$espacio_efectivo

for (iteracion in seq_len(PARAM$hyperparametertuning$n_trials)) {
  trial <- estudio$ask()

  # Si falla una propuesta o la evaluación, marco el ensayo como fallido
  # y detengo la búsqueda mostrando el error original.
  resultado <- tryCatch({
    x <- list(
      learning_rate = trial$suggest_float(
        "learning_rate", espacio$learning_rate[1],
        espacio$learning_rate[2], log = TRUE
      ),
      num_leaves = trial$suggest_int(
        "num_leaves", as.integer(espacio$num_leaves[1]),
        as.integer(espacio$num_leaves[2]), log = TRUE
      ),
      min_data_in_leaf = trial$suggest_int(
        "min_data_in_leaf", as.integer(espacio$min_data_in_leaf[1]),
        as.integer(espacio$min_data_in_leaf[2]), log = TRUE
      ),
      feature_fraction = trial$suggest_float(
        "feature_fraction", espacio$feature_fraction[1],
        espacio$feature_fraction[2]
      ),
      bagging_fraction = trial$suggest_float(
        "bagging_fraction", espacio$bagging_fraction[1],
        espacio$bagging_fraction[2]
      )
    )

    evaluacion <- Estimar_AUC_lightgbm(x)
    list(
      parametros = c(x, list(num_iterations = evaluacion$num_iterations)),
      AUC = evaluacion$AUC
    )
  }, error = function(e) {
    invisible(estudio$tell(trial, state = estado_fallido))
    stop(e)
  })

  # num_iterations queda fijo en 1000 y no se sugiere como hiperparámetro.
  trial$set_user_attr("num_iterations", as.integer(resultado$parametros$num_iterations))
  invisible(estudio$tell(trial, as.numeric(resultado$AUC)))

  tb_nueva <- rbindlist(list(
    tb_nueva,
    as.data.table(c(resultado$parametros, list(AUC = resultado$AUC)))
  ), use.names = TRUE)

  # Actualizo el historial en disco tras cada evaluación exitosa.
  fwrite(tb_nueva, file = "tb_optuna_AUC_BAJA2_1000.txt", sep = "\t", append = FALSE)

  message("Ensayo ", iteracion, "/", PARAM$hyperparametertuning$n_trials,
    " | mejor AUC: ", signif(estudio$best_value, 7)
  )
}

Fri Oct 23 16:05:28 2026  0.132341261644285, 95, 217, 0.778609581661164, 0.967928475119342 AUC en ronda 1000: 0.872588705603389

Ensayo 1/80 | mejor AUC: 0.8725887

Fri Oct 23 16:06:18 2026  0.0812689312595428, 65, 365, 0.950423018005721, 0.914980212234118 AUC en ronda 1000: 0.889976781142474

Ensayo 2/80 | mejor AUC: 0.8899768

Fri Oct 23 16:09:02 2026  0.149008218405735, 174, 59, 0.982498572682648, 0.658233279024736 AUC en ronda 1000: 0.875230789281519

Ensayo 4/80 | mejor AUC: 0.9077827

Fri Oct 23 16:10:03 2026  0.0849151036985228, 97, 21, 0.539686926118219, 0.893435239903227 AUC en ronda 1000: 0.887377419042796

Ensayo 5/80 | mejor AUC: 0.9077827

Fri Oct 23 16:10:51 2026  0.0683699094230481, 84, 37, 0.915984494337731, 0.815219963489258 AUC en ronda 1000: 0.889676247751181

Ensayo 6/80 | mejor AUC: 0.9077827

Fri Oct 23 16:11:47 2026  0.0130155017633448, 89, 44, 0.526872158765736, 0.699648444159745 AUC en ronda 1000: 0.908652506707846

Ensayo 7/80 | mejor AUC: 0.9086525

Fri Oct 2

In [25]:
# Veo los hiperparámetros y la AUC de los ensayos completados.
tb_nueva

learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,num_iterations,AUC
<dbl>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>
0.13234126,95,217,0.7786096,0.9679285,1000,0.8725887
0.08126893,65,365,0.9504230,0.9149802,1000,0.8899768
0.01249456,254,24,0.5939015,0.9541372,1000,0.9077827
0.14900822,174,59,0.9824986,0.6582333,1000,0.8752308
0.08491510,97,21,0.5396869,0.8934352,1000,0.8873774
0.06836991,84,37,0.9159845,0.8152200,1000,0.8896762
0.01301550,89,44,0.5268722,0.6996484,1000,0.9086525
0.09365326,71,1136,0.7677841,0.8935703,1000,0.8927881
0.03169860,106,1099,0.5915013,0.6959853,1000,0.9037602


In [26]:
# Guardo el historial completo, sin duplicar filas al reejecutar la celda.
fwrite(tb_nueva,
  file = "tb_optuna_AUC_BAJA2_1000.txt",
  sep = "\t",
  append = FALSE
)

In [27]:
# Selecciono el mejor ensayo y conservo la interfaz para producción.
setorder(tb_nueva, -AUC)
PARAM$out$lgbm$AUC <- as.numeric(estudio$best_value)
PARAM$out$lgbm$mejores_hiperparametros <- modifyList(
  estudio$best_params,
  list(num_iterations = as.integer(PARAM$hyperparametertuning$num_iterations))
)
PARAM$out$lgbm$mejores_hiperparametros

$learning_rate
[1] 0.01001333

$num_leaves
[1] 19

$min_data_in_leaf
[1] 591

$feature_fraction
[1] 0.5311985

$bagging_fraction
[1] 0.7267378

$num_iterations
[1] 1000

In [28]:
write_yaml( PARAM, file="PARAM.yml")

## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [29]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparametros

In [30]:
# Misma etiqueta que en validación: solo BAJA+2 es positiva.
# Las filas futuras sin clase conservan NA; no se usan como entrenamiento.
dataset[, clase01 := as.integer(clase_ternaria == "BAJA+2")]

In [31]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [32]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [33]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 149911

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 0.7267378

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 1

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 1000

$learning_rate
[1] 0.01001333

$feature_fraction
[1] 0.5311985

$num_leaves
[1] 19

$min_data_in_leaf
[1] 591

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [34]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [35]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [36]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [37]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [38]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [39]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [40]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

Warning message in dir.create("kaggle"):
“'kaggle' already exists”


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-04**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-04**

Siéntase libre de agregar columnas a la hoja **TareaHogar-04**  en caso de ser necesario.